# Section 2.5 — Tree-SHAP-guided *de novo* peptide generation (ProtGPT2)
## *Klebsiella pneumoniae* AMP pipeline · PepFoundry + RDKit-style charge + modlAMP moment

| Step | Proposal | Cell |
|---|---|---|
| Install + imports | — | 1–2 |
| Load `kpneumoniae_features_protgpt2.csv` (1000 rows, 30 features) | 2.2 | 3 |
| Retrain 2.4 master gate (benchmark + Youden threshold) | 2.4.3/2.4.4 | 4 |
| TreeSHAP pattern table | 2.4.4 → 2.5.2 | 5 |
| SHAP-guided filter rules (layer-1 + layer-2) | 2.5.2 | 6 |
| ProtGPT2 generation from top active templates | 2.5.1 | 7 |
| QC + physicochemical filter + novelty check | 2.5.2/2.5.3 | 8 |
| Re-evaluation with frozen gate → shortlist for 2.6/2.7/2.8 | 2.5.4 | 9 |

> **Toolchain (your 2.3/2.4 substitution, verified on this CSV):**
> `pf_N` = PepFoundry pooled index N (exact); `hydrophobic_moment` = modlAMP Eisenberg (exact);
> `net_charge` = pH-7 titration, amidated C-term (≈ CSV within 0.01–0.2).
>
> **Runtime:** CPU is fine. ProtGPT2 download is ~500 MB on first run.
> For a 2-minute check set `SMOKE_TEST = True` (template mutants, no download).


In [ ]:
# @title 1. Install packages
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("pandas", "numpy", "scikit-learn", "matplotlib", "joblib", "biopython")
_pip("rdkit", "openbabel-wheel")
_pip("git+https://github.com/BilodeauGroup/PepFoundry.git")
_pip("torch", "transformers", "modlamp", "shap", "xgboost", "lightgbm")
import importlib
for m in ("pandas", "sklearn", "Bio", "torch", "transformers", "shap", "modlamp"):
    try:
        print(f"{m:14s} OK  {getattr(importlib.import_module(m), '__version__', '?')}")
    except Exception as e:
        print(f"{m:14s} FAIL {str(e)[:100]}")
try:
    from pepfoundry.interface import PepFoundry
    print("pepfoundry     OK (first build of the residue library takes ~1 min)")
except Exception as e:
    print("pepfoundry     FAIL:", repr(e)[:200])


In [ ]:
# @title 2. Imports + configuration
import os, re
import numpy as np
import pandas as pd
SEED = 42
np.random.seed(SEED)
MIN_LEN, MAX_LEN = 10, 30   # proposal 2.2.2
NOVELTY_IDENTITY = 0.80     # proposal 2.2.2 CD-HIT cutoff, reused in 2.5.3
PROB_NOMINAL = 0.70         # proposal 2.5.4 nominal threshold
PROTGPT2_ID = "nferruz/ProtGPT2"
VALID_AA = re.compile(r"^[ACDEFGHIKLMNPQRSTVWY]+$")
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
PKA = {"N_term": 8.60, "K": 10.53, "R": 12.48, "H": 6.04,
       "D": 3.90, "E": 4.07, "C": 8.37, "Y": 10.46}
# generation size — raise for the real run (e.g. 8 x 60)
N_TEMPLATES, PER_TEMPLATE, FREE_RUN = 5, 20, 0
SMOKE_TEST = False  # set True for a fast no-download check
OUT = "/content/amp_25_out"
os.makedirs(OUT, exist_ok=True)
print("outputs ->", OUT)


In [ ]:
# @title 3. Load kpneumoniae_features_protgpt2.csv
import glob
try:
    from google.colab import files
    HAVE_COLAB = True
except ImportError:
    HAVE_COLAB = False
cands = glob.glob("kpneumoniae_features_protgpt2.csv") + glob.glob("/content/*.csv")
if not cands and HAVE_COLAB:
    print("Upload kpneumoniae_features_protgpt2.csv:")
    files.upload()
    cands = glob.glob("*.csv")
CSV_PATH = cands[0]
print("CSV:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
df["sequence"] = df["sequence"].astype(str).str.strip().str.upper()
PF_COLS = [c for c in df.columns if c.startswith("pf_")]
FEATURE_COLS = ["net_charge", "hydrophobic_moment"] + PF_COLS
PF_INDICES = [int(c.split("_")[1]) for c in PF_COLS]
print(f"{len(df)} rows ({int(df['label'].sum())} active / {int((1-df['label']).sum())} inactive), "
      f"{len(FEATURE_COLS)} features; length range {df['sequence'].str.len().min()}-{df['sequence'].str.len().max()}")
df.head(3)


In [ ]:
# @title 4. Physicochemistry + PepFoundry helpers (same pipeline as training)
def net_charge_hh(seq, pH=7.0, amide=True):
    """Fractional net charge at pH 7, amidated C-term (synthetic AMPs)."""
    q = 1.0 / (1.0 + 10.0 ** (pH - PKA["N_term"]))
    if not amide:
        q += -1.0 / (1.0 + 10.0 ** (3.55 - pH))
    for aa in ("K", "R", "H"):
        q += seq.count(aa) / (1.0 + 10.0 ** (pH - PKA[aa]))
    for aa in ("D", "E", "C", "Y"):
        q += -seq.count(aa) / (1.0 + 10.0 ** (PKA[aa] - pH))
    return float(q)

def hydrophobic_moment_modlamp(seq):
    from modlamp.descriptors import PeptideDescriptor
    d = PeptideDescriptor(seq, "Eisenberg")
    d.calculate_moment()
    return float(np.asarray(d.descriptor).ravel()[0])

_PF = None
def pf_engine():
    global _PF
    if _PF is None:
        from pepfoundry.interface import PepFoundry
        _PF = PepFoundry()
    return _PF

def pf_embed160(seq):
    mol = pf_engine().get_peptide(seq, plot_peptide=False)
    a_t, b_t = pf_engine().get_peptide_tensor_atomic_features(mol)
    a, b = np.asarray(a_t, float), np.asarray(b_t, float)
    def agg(t):
        return np.concatenate([t.mean(0), t.std(0), t.max(0), t.sum(0)])
    return np.concatenate([agg(a), agg(b)])

def featurise(seqs):
    return np.array([[net_charge_hh(s), hydrophobic_moment_modlamp(s)] +
                     [float(v) for v in pf_embed160(s)[PF_INDICES]] for s in seqs])

# sanity: must reproduce the CSV row exactly for pf_* and moment
t = df.iloc[0]
e = pf_embed160(t["sequence"])
print("pf_26  csv=%.6f recomputed=%.6f" % (t["pf_26"], e[26]))
print("moment csv=%.6f recomputed=%.6f" % (t["hydrophobic_moment"], hydrophobic_moment_modlamp(t["sequence"])))
print("charge csv=%.4f recomputed=%.4f" % (t["net_charge"], net_charge_hh(t["sequence"])))


In [ ]:
# @title 5. Retrain 2.4 master gate (benchmark + Youden threshold)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import matthews_corrcoef, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
X = df[FEATURE_COLS].to_numpy(float); y = df["label"].to_numpy(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
models = {
    "LogReg": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "SVM-RBF": SVC(kernel="rbf", class_weight="balanced", random_state=SEED),
    "DecTree": DecisionTreeClassifier(random_state=SEED, class_weight="balanced"),
    "RandForest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                         n_jobs=-1, random_state=SEED)}
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                      subsample=0.9, colsample_bytree=0.9,
                                      eval_metric="logloss", n_jobs=-1, random_state=SEED)
except Exception: pass
try:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier(n_estimators=300, verbose=-1, random_state=SEED)
except Exception: pass
rows = []
for name, clf in models.items():
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    rows.append({"model": name, "MCC": matthews_corrcoef(yte, (p >= 0.5).astype(int)),
                 "AUROC": roc_auc_score(yte, p)})
bench = pd.DataFrame(rows).sort_values("MCC", ascending=False).reset_index(drop=True)
print(bench.round(4).to_string(index=False))
gs = GridSearchCV(RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
                  {"n_estimators": [300, 500], "max_depth": [None, 12, 20],
                   "min_samples_split": [2, 5]},
                  cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                  scoring="roc_auc", n_jobs=-1)
gs.fit(Xtr, ytr)
gate = gs.best_estimator_
print("tuned RF:", gs.best_params_, "CV AUC=%.4f" % gs.best_score_)
fpr, tpr, thr = roc_curve(ytr, gate.predict_proba(Xtr)[:, 1])
YOUDEN = float(thr[int(np.argmax(tpr - fpr))])
pte = gate.predict_proba(Xte)[:, 1]
print(f"Youden={YOUDEN:.4f} | test MCC={(matthews_corrcoef(yte, (pte>=YOUDEN).astype(int))):.4f} "
      f"AUROC={roc_auc_score(yte, pte):.4f}")
import joblib
joblib.dump({"pipeline": gate, "threshold_youden": YOUDEN, "feature_cols": FEATURE_COLS},
            f"{OUT}/02_4_master_gate.joblib")
bench.to_csv(f"{OUT}/02_4_benchmark.csv", index=False)


In [ ]:
# @title 6. TreeSHAP pattern table (2.5.2 rule source)
import shap, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
Xa = df[FEATURE_COLS].to_numpy(float); ya = df["label"].to_numpy(int)
sv = np.asarray(shap.TreeExplainer(gate).shap_values(Xa))
sv = sv[:, :, -1] if sv.ndim == 3 else np.asarray(sv[-1] if isinstance(sv, list) else sv)
patterns = pd.DataFrame({
    "descriptor": FEATURE_COLS,
    "mean_shap_active": [sv[ya == 1, j].mean() for j in range(sv.shape[1])],
    "mean_shap_inactive": [sv[ya == 0, j].mean() for j in range(sv.shape[1])],
    "mean_abs_shap": np.abs(sv).mean(axis=0),
    "active_mean": [Xa[ya == 1, j].mean() for j in range(Xa.shape[1])],
    "inactive_mean": [Xa[ya == 0, j].mean() for j in range(Xa.shape[1])]})
patterns["shap_gap"] = patterns["mean_shap_active"] - patterns["mean_shap_inactive"]
patterns["favours"] = np.where(patterns["shap_gap"] > 0, "Active", "Inactive")
patterns["direction"] = np.where(patterns["active_mean"] > patterns["inactive_mean"],
                                 "higher favours Active", "lower favours Active")
patterns = patterns.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
patterns.to_csv(f"{OUT}/02_5_shap_patterns.csv", index=False)
print(patterns.head(12).round(4).to_string(index=False))
order = np.argsort(np.abs(sv).mean(0))[::-1][:15]
plt.figure(figsize=(8, 5))
plt.barh(np.arange(len(order))[::-1], np.abs(sv).mean(0)[order])
plt.yticks(np.arange(len(order))[::-1], np.asarray(FEATURE_COLS)[order])
plt.xlabel("mean |SHAP|"); plt.title("Top drivers of K. pneumoniae activity (TreeSHAP)")
plt.tight_layout(); plt.savefig(f"{OUT}/shap_top_features.png", dpi=200, bbox_inches="tight"); plt.close()
plt.figure(figsize=(10, 7))
shap.summary_plot(sv, Xa, feature_names=FEATURE_COLS, show=False)
plt.tight_layout(); plt.savefig(f"{OUT}/shap_summary.png", dpi=200, bbox_inches="tight"); plt.close()


In [ ]:
# @title 7. Filter rules — layer-1 (proposal gates) + layer-2 (SHAP)
act = df[df["label"].astype(int) == 1]
def band(col, default):
    v = pd.to_numeric(act[col], errors="coerce").dropna()
    return (float(v.quantile(0.10)), float(v.quantile(0.90))) if len(v) >= 10 else default
c_lo, _ = band("net_charge", (1.0, 12.0)); m_lo, _ = band("hydrophobic_moment", (0.20, 0.80))
rules = [
    {"layer": 1, "descriptor": "length", "rule": "10<=L<=30", "lo": MIN_LEN, "hi": MAX_LEN,
     "rationale": "proposal 2.2.2 synthetic-viability window"},
    {"layer": 1, "descriptor": "net_charge", "rule": "ge", "lo": max(0.0, c_lo), "hi": None,
     "rationale": f"cationic attraction; active 10th pct={c_lo:.2f}"},
    {"layer": 1, "descriptor": "hydrophobic_moment", "rule": "ge", "lo": m_lo, "hi": None,
     "rationale": f"amphipathic insertion; active 10th pct={m_lo:.3f}"}]
cut = float(patterns["mean_abs_shap"].quantile(0.90))
for _, r in patterns[patterns["mean_abs_shap"] >= cut].iterrows():
    if r["descriptor"] in ("net_charge", "hydrophobic_moment"):
        continue
    op = ("ge" if r["direction"].startswith("higher") else "le") if r["favours"] == "Active" \
         else ("le" if r["direction"].startswith("higher") else "ge")
    rules.append({"layer": 2, "descriptor": r["descriptor"], "rule": op,
                  "lo": float(r["active_mean"]) if op == "ge" else None,
                  "hi": float(r["active_mean"]) if op == "le" else None,
                  "rationale": f"{r['direction']} (SHAP gap {r['shap_gap']:+.3f})"})
rules = pd.DataFrame(rules)
rules.to_csv(f"{OUT}/02_5_filter_rules.csv", index=False)
print(f"layer-1={(rules['layer']==1).sum()} layer-2={(rules['layer']==2).sum()}")
print(rules.to_string(index=False))

def passes(row):
    for _, r in rules.iterrows():
        d = r["descriptor"]
        v = float(row.get("length", len(row.get("sequence", "")))) if d == "length" else row.get(d, np.nan)
        if d != "length" and (v is None or not np.isfinite(float(v))):
            continue
        v = float(v)
        if r["rule"] in ("ge", "10<=L<=30") and r["lo"] is not None and v < float(r["lo"]):
            return False, f"{d}={v:.3f} < {float(r['lo']):.3f}"
        if r["rule"] == "le" and r["hi"] is not None and v > float(r["hi"]):
            return False, f"{d}={v:.3f} > {float(r['hi']):.3f}"
    return True, ""


In [ ]:
# @title 8. ProtGPT2 generation (2.5.1)
tmp = df[df["sequence"].astype(str).str.match(VALID_AA.pattern)]
tmp = tmp[tmp["label"].astype(int) == 1].copy()
tmp["p_active"] = gate.predict_proba(tmp[FEATURE_COLS].to_numpy(float))[:, 1]
templates = tmp.drop_duplicates("sequence").sort_values("p_active", ascending=False)["sequence"].tolist()[:N_TEMPLATES]
print("seed templates:", templates)
if SMOKE_TEST:
    rng = np.random.default_rng(SEED); aas = list("ACDEFGHIKLMNPQRSTVWY"); raw = []
    for t in templates:
        for _ in range(20):
            s = list(t)
            for _ in range(rng.integers(3, 7)):
                op = rng.choice(["sub", "sub", "sub", "ins", "del"]); i = int(rng.integers(0, len(s)))
                if op == "sub": s[i] = rng.choice(aas)
                elif op == "ins" and len(s) < MAX_LEN: s.insert(i, rng.choice(aas))
                elif op == "del" and len(s) > MIN_LEN: s.pop(i)
            raw.append("".join(s))
    print(f"[smoke] {len(raw)} template mutants (no ProtGPT2 download)")
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print(f"loading {PROTGPT2_ID} (~500 MB first run) ...")
    tok = AutoTokenizer.from_pretrained(PROTGPT2_ID)
    lm = AutoModelForCausalLM.from_pretrained(PROTGPT2_ID); lm.eval()
    torch.manual_seed(SEED)
    bos = tok("<|endoftext|>", add_special_tokens=False).input_ids
    raw = []
    for t in templates:
        with torch.no_grad():
            gen = lm.generate(torch.tensor([bos + tok(" ".join(list(t)), add_special_tokens=False).input_ids]),
                              max_new_tokens=MAX_LEN + 10, do_sample=True, top_k=950,
                              temperature=0.8, repetition_penalty=1.2,
                              num_return_sequences=PER_TEMPLATE)
        for g in gen:
            s = "".join(ch for ch in tok.decode(g, skip_special_tokens=True).upper() if ch in STANDARD_AA)
            raw.append(s[len(t):] if t and s.startswith(t) else s)
        print(f"  {t[:12]:12s} -> {len(gen)} decoded")
    if FREE_RUN:
        with torch.no_grad():
            gen = lm.generate(torch.tensor([bos]), max_new_tokens=MAX_LEN + 10, do_sample=True,
                              top_k=950, temperature=0.8, repetition_penalty=1.2,
                              num_return_sequences=FREE_RUN)
        for g in gen:
            raw.append("".join(ch for ch in tok.decode(g, skip_special_tokens=True).upper() if ch in STANDARD_AA))
    print(f"[2.5.1] {len(raw)} raw sequences")
pd.DataFrame({"sequence": raw, "length": [len(s) for s in raw],
              "is_canonical": [bool(VALID_AA.match(s)) for s in raw]}).to_csv(f"{OUT}/02_5_1_generated_raw.csv", index=False)


In [ ]:
# @title 9. QC + 2.5.2 filter + 2.5.3 novelty + 2.5.4 re-evaluation → shortlist
from difflib import SequenceMatcher
qc = [s for s in dict.fromkeys(raw) if VALID_AA.match(s) and MIN_LEN <= len(s) <= MAX_LEN]
print(f"[2.5.3] {len(qc)}/{len(raw)} survive canonical + {MIN_LEN}-{MAX_LEN} length + dedupe")
Xg = featurise(qc)
cand = pd.DataFrame(Xg, columns=FEATURE_COLS)
cand.insert(0, "sequence", qc); cand["length"] = [len(s) for s in qc]
audit = [passes(cand.iloc[i]) for i in range(len(qc))]
cand["passes_2_5_2"] = [ok for ok, _ in audit]; cand["failed_rule"] = [w for _, w in audit]
cand[["sequence", "passes_2_5_2", "failed_rule"]].to_csv(f"{OUT}/02_5_2_filter_audit.csv", index=False)
print(f"[2.5.2] {int(cand['passes_2_5_2'].sum())}/{len(cand)} pass physicochemical + SHAP filters")
kept = cand[cand["passes_2_5_2"]].reset_index(drop=True)
refs = df["sequence"].astype(str).tolist()
mi, hit = [], []
for s in kept["sequence"]:
    b, br = 0.0, ""
    for r in refs:
        if abs(len(s) - len(r)) / max(len(s), len(r)) > (1 - NOVELTY_IDENTITY): continue
        v = SequenceMatcher(None, s, r).ratio()
        if v > b: b, br = v, r
    mi.append(b); hit.append(br)
kept["max_identity_to_known"] = mi; kept["nearest_known_peptide"] = hit
kept.to_csv(f"{OUT}/02_5_3_novelty.csv", index=False)
novel = kept[kept["max_identity_to_known"] <= NOVELTY_IDENTITY].reset_index(drop=True)
print(f"[2.5.3] {len(novel)}/{len(kept)} novel (<={100*NOVELTY_IDENTITY:.0f}% identity)")
novel["p_active"] = gate.predict_proba(novel[FEATURE_COLS].to_numpy(float))[:, 1]
novel["threshold_youden"] = YOUDEN
novel["passes_youden"] = novel["p_active"] >= YOUDEN
novel["passes_nominal_0.70"] = novel["p_active"] >= PROB_NOMINAL
novel = novel.sort_values("p_active", ascending=False).reset_index(drop=True)
novel.to_csv(f"{OUT}/02_5_4_candidate_scores.csv", index=False)
shortlist = novel[novel["p_active"] >= PROB_NOMINAL].copy()
shortlist.to_csv(f"{OUT}/02_5_shortlist.csv", index=False)
print(novel[["sequence", "p_active", "net_charge", "hydrophobic_moment",
             "max_identity_to_known"]].head(15).round(3).to_string(index=False))
print(f"\nYouden {YOUDEN:.3f} -> {int(novel['passes_youden'].sum())} | nominal 0.70 -> "
      f"{int(novel['passes_nominal_0.70'].sum())} | SHORTLIST: {len(shortlist)} → 02_5_shortlist.csv")


In [ ]:
# @title 10. Download results
import glob
print("\n".join(sorted(glob.glob(f"{OUT}/*"))))
try:
    from google.colab import files
    for f in ["02_5_shortlist.csv", "02_5_4_candidate_scores.csv",
              "02_5_shap_patterns.csv", "02_5_filter_rules.csv"]:
        files.download(f"{OUT}/{f}")
except ImportError:
    print("not on Colab — files are in", OUT)
